In [16]:
import os
import json

import numpy as np
import pandas as pd

In [17]:
runs = [
    os.path.join('experiments', str(x)) for x in range(1616, 1656)
]

assert len(runs) == 40, len(runs)

In [18]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [19]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # print(log['training_time'].cumsum().max())
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    # first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    # info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()
    
    df.loc[len(df)] = info

In [20]:
results = df[[
    'soln_name',
    # 'time_to_5p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed'
]].rename(columns={
    # 'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to min. RL2E. (s)       Min. RL2E. (%)
Solution        w/ BC                                            
Hopf fibration  False            165.5 ± 7.9  1.13e-02 ± 3.02e-04
                True            582.8 ± 10.3  1.62e-02 ± 6.11e-04
Plane waves     False           477.1 ± 46.8  1.18e-02 ± 6.97e-04
                True            551.2 ± 14.3  1.13e-02 ± 6.57e-04
Radial waves    False           577.7 ± 10.2  1.91e-02 ± 6.87e-04
                True             596.2 ± 1.4  5.43e-02 ± 8.07e-04
Random solution False           446.6 ± 78.4  1.21e-02 ± 4.35e-04
                True             596.4 ± 2.2  5.68e-02 ± 1.47e-03

In [21]:
print(display_results.to_latex())

\begin{tabular}{llll}
\toprule
                &       & Time to min. RL2E. (s) &       Min. RL2E. (\%) \\
Solution & w/ BC &                        &                      \\
\midrule
Hopf fibration & False &            165.5 ± 7.9 &  1.13e-02 ± 3.02e-04 \\
                & True  &           582.8 ± 10.3 &  1.62e-02 ± 6.11e-04 \\
Plane waves & False &           477.1 ± 46.8 &  1.18e-02 ± 6.97e-04 \\
                & True  &           551.2 ± 14.3 &  1.13e-02 ± 6.57e-04 \\
Radial waves & False &           577.7 ± 10.2 &  1.91e-02 ± 6.87e-04 \\
                & True  &            596.2 ± 1.4 &  5.43e-02 ± 8.07e-04 \\
Random solution & False &           446.6 ± 78.4 &  1.21e-02 ± 4.35e-04 \\
                & True  &            596.4 ± 2.2 &  5.68e-02 ± 1.47e-03 \\
\bottomrule
\end{tabular}



/var/folders/54/ctkh4lg97kd4x1fv3qzrpd3c0000gp/T/ipykernel_8176/2555358579.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(display_results.to_latex())
